# 面试问题：FP8 的 E4M3、E5M2、缩放因子与 delayed scaling 怎样实现？

**回答主线。** FP8 不是把 FP16 张量直接改成 8 bit 就结束。E4M3 用更多尾数换精度，常用于前向激活和权重；E5M2 用更多指数换动态范围，更适合梯度。训练系统还要为每个张量或通道维护 scale、amax 历史、溢出率和高精度主副本。下面实现一个明确标注为“数值语义教学版”的 FP8 仿真器：它模拟舍入、下溢、饱和和反缩放，但不冒充 GPU 的逐 bit 编码或真实 Tensor Core kernel。

面试时应先区分三层：格式能表达什么、缩放策略把业务张量映射到哪里、硬件 kernel 是否真正加速。只证明误差小，不能推出训练稳定或吞吐提升。


In [ ]:
import hashlib, json, math
from dataclasses import dataclass
import numpy as np

# 固定随机种子，保证所有误差与门禁断言可重复。
rng151 = np.random.default_rng(151)

def rel_rmse151(reference, candidate):
    reference = np.asarray(reference, dtype=np.float64)
    candidate = np.asarray(candidate, dtype=np.float64)
    denom = max(float(np.sqrt(np.mean(reference ** 2))), 1e-12)
    return float(np.sqrt(np.mean((reference - candidate) ** 2)) / denom)

probe151 = rng151.normal(size=(8, 4))
assert probe151.shape == (8, 4)
assert rel_rmse151(probe151, probe151) == 0.0
assert np.isfinite(probe151).all()


## 1. 格式决定动态范围与相邻可表示数间距

真实 E4M3/E5M2 对 NaN、Infinity 和边界编码有专门规范。教学实现直接使用两种格式的最小正规数、最大有限值和尾数位数，按当前指数计算量化步长；绝对值更小时进入 subnormal 网格，更大时饱和。这样可以观察数值行为，但序列化模型权重时必须使用硬件/框架认可的格式实现。


In [ ]:
@dataclass(frozen=True)
class FP8Format151:
    name: str
    mantissa_bits: int
    min_normal: float
    max_finite: float

E4M3_151 = FP8Format151("E4M3", 3, 2.0 ** -6, 448.0)
E5M2_151 = FP8Format151("E5M2", 2, 2.0 ** -14, 57344.0)

def fp8_quantize151(x, fmt):
    # 用指数相关步长舍入，并显式处理 subnormal、零与饱和。
    x = np.asarray(x, dtype=np.float64)
    sign, ax = np.sign(x), np.abs(x)
    safe = np.maximum(ax, np.finfo(np.float64).tiny)
    exponent = np.floor(np.log2(safe))
    normal_step = np.exp2(exponent - fmt.mantissa_bits)
    normal = np.round(ax / normal_step) * normal_step
    sub_step = fmt.min_normal / (2 ** fmt.mantissa_bits)
    subnormal = np.round(ax / sub_step) * sub_step
    magnitude = np.where(ax < fmt.min_normal, subnormal, normal)
    magnitude = np.where(ax == 0.0, 0.0, np.minimum(magnitude, fmt.max_finite))
    return sign * magnitude

values151 = np.array([0.0, 1e-4, 1.0, 1.1, 1000.0])
q4_151 = fp8_quantize151(values151, E4M3_151)
q5_151 = fp8_quantize151(values151, E5M2_151)
assert q4_151[-1] == E4M3_151.max_finite
assert q5_151[-1] < E5M2_151.max_finite
assert q4_151[0] == 0.0 and np.isfinite(q5_151).all()


## 2. Scale 把张量动态范围映射到 FP8 可用区间

常见语义是 `q = cast(x * scale)`，计算前再用 `q / scale` 恢复近似值。margin 为峰值预留指数空间，能降低下一步突然放大时的饱和风险，但会牺牲小数分辨率。零张量必须返回有限 scale；否则一次空梯度就可能把 NaN 写进状态。


In [ ]:
def choose_scale151(amax, fmt, margin=0):
    # margin 每增加 1，就为未来峰值预留一倍动态范围。
    amax = float(amax)
    return 1.0 if amax <= 0 else fmt.max_finite / (amax * (2.0 ** margin))

def scaled_cast151(x, fmt, scale):
    return fp8_quantize151(np.asarray(x) * scale, fmt) / scale

x151 = np.array([-30.0, -1.25, 0.0, 2.5, 30.0])
scale151 = choose_scale151(np.max(np.abs(x151)), E4M3_151, margin=1)
restored151 = scaled_cast151(x151, E4M3_151, scale151)
assert scale151 > 1.0
assert np.max(np.abs(restored151)) <= 30.0 + 1e-9
assert choose_scale151(0.0, E4M3_151) == 1.0


## 3. Per-tensor 与 per-channel scale 是元数据—误差权衡

单个 scale 成本最低，但一个异常通道会占满动态范围。按列 scale 能保护量级很小的通道，却需要更多元数据与 kernel 支持。训练配置必须写清归约轴；“per-channel”若没有张量布局，无法复现。


In [ ]:
def scaled_cast_axis151(x, fmt, reduce_axis):
    # keepdims 保留 scale 的广播轴，避免静默缩错维度。
    x = np.asarray(x, dtype=np.float64)
    amax = np.max(np.abs(x), axis=reduce_axis, keepdims=True)
    scales = np.where(amax > 0, fmt.max_finite / amax, 1.0)
    return fp8_quantize151(x * scales, fmt) / scales, scales

base151 = rng151.normal(size=(128, 4)) * np.array([0.01, 0.1, 1.0, 100.0])
tensor_scale151 = choose_scale151(np.max(np.abs(base151)), E4M3_151)
per_tensor151 = scaled_cast151(base151, E4M3_151, tensor_scale151)
per_channel151, scales_channel151 = scaled_cast_axis151(base151, E4M3_151, reduce_axis=0)
assert scales_channel151.shape == (1, 4)
assert rel_rmse151(base151, per_channel151) < rel_rmse151(base151, per_tensor151)
assert np.isfinite(per_channel151).all()


## 4. Delayed scaling 用 amax 历史降低 scale 抖动

每步都用当前峰值更新是 dynamic scaling；delayed scaling 每隔若干步读取一段 amax 历史。历史窗口能吸收正常波动，但旧异常值停留过久会浪费精度。checkpoint 必须保存 history、当前位置和当前 scale，而不只是模型参数。


In [ ]:
class DelayedScaler151:
    def __init__(self, fmt, history_len=4, interval=2, margin=1):
        self.fmt, self.history_len = fmt, history_len
        self.interval, self.margin = interval, margin
        self.history, self.step, self.scale = [], 0, 1.0

    def observe(self, x):
        # 先记录 amax；只有到更新步才改变正在使用的 scale。
        self.step += 1
        self.history.append(float(np.max(np.abs(x))))
        self.history = self.history[-self.history_len:]
        if self.step % self.interval == 0:
            self.scale = choose_scale151(max(self.history), self.fmt, self.margin)
        return self.scale

    def state_dict(self):
        return {"history": list(self.history), "step": self.step, "scale": self.scale}

scaler151 = DelayedScaler151(E4M3_151, history_len=3, interval=2)
s1_151 = scaler151.observe(np.array([1.0]))
s2_151 = scaler151.observe(np.array([4.0]))
s3_151 = scaler151.observe(np.array([2.0]))
assert s1_151 == 1.0 and s2_151 != s1_151
assert s3_151 == s2_151
assert scaler151.state_dict()["history"] == [1.0, 4.0, 2.0]


## 5. 饱和率与 underflow 率比单一 MSE 更可诊断

Scale 太大导致大值饱和，太小则把小值冲到零。离线应至少记录饱和、零化、相对误差和分位数切片。均值误差正常并不代表稀有大梯度未被截断。


In [ ]:
def cast_diagnostics151(x, fmt, scale):
    # 在缩放后的格式域统计饱和和非零值被量化为零的比例。
    x = np.asarray(x, dtype=np.float64)
    scaled = x * scale
    q = fp8_quantize151(scaled, fmt)
    sat = np.mean(np.abs(scaled) > fmt.max_finite)
    under = np.mean((x != 0) & (q == 0))
    return {"saturation": float(sat), "underflow": float(under), "rel_rmse": rel_rmse151(x, q / scale)}

mixed151 = np.array([1e-8, 1e-5, 0.1, 1.0, 1000.0])
diag_tight151 = cast_diagnostics151(mixed151, E4M3_151, scale=1.0)
diag_small151 = cast_diagnostics151(mixed151, E4M3_151, scale=0.1)
assert diag_tight151["saturation"] > 0
assert diag_small151["saturation"] == 0
assert 0.0 <= diag_small151["underflow"] <= 1.0


## 6. FP8 GEMM 仍用高精度累加与主权重

下面只量化线性层的输入和权重，再以 FP32 模拟累加。真实训练通常保留 BF16/FP16 或 FP32 主权重，用低精度副本参与 GEMM。若直接在 FP8 权重上累积微小更新，更新可能永久小于一个量化台阶。


In [ ]:
def fp8_linear151(x, weight, fmt=E4M3_151):
    # 输入和权重独立缩放，矩阵乘的累加语义保留为 float32。
    sx = choose_scale151(np.max(np.abs(x)), fmt, margin=1)
    sw = choose_scale151(np.max(np.abs(weight)), fmt, margin=1)
    xq = scaled_cast151(x, fmt, sx).astype(np.float32)
    wq = scaled_cast151(weight, fmt, sw).astype(np.float32)
    return xq @ wq, {"x_scale": sx, "w_scale": sw}

xlin151 = rng151.normal(size=(16, 12))
wlin151 = rng151.normal(size=(12, 7))
ref151 = xlin151 @ wlin151
out151, meta151 = fp8_linear151(xlin151, wlin151)
assert out151.shape == (16, 7)
assert rel_rmse151(ref151, out151) < 0.12
assert meta151["x_scale"] > 0 and meta151["w_scale"] > 0


## 7. E5M2 的价值是梯度动态范围，不是处处更准

在同一未缩放探针上，E5M2 能保留 E4M3 会饱和的大梯度；但 E4M3 的额外尾数位通常让中等量级值更细。常见 recipe 因而对前向与反向选择不同格式，并对异常梯度单独监控。


In [ ]:
gradient151 = np.array([2.0 ** p for p in range(-18, 16)], dtype=np.float64)
grad4_151 = fp8_quantize151(gradient151, E4M3_151)
grad5_151 = fp8_quantize151(gradient151, E5M2_151)
# 比较格式本身的范围；实际训练还会先应用张量 scale。
assert grad4_151[-1] == E4M3_151.max_finite
assert grad5_151[-1] > grad4_151[-1]
assert np.count_nonzero(grad5_151) >= np.count_nonzero(grad4_151)


## 8. Recipe 是需要版本和回归门禁的训练制品

格式、归约轴、history、margin、更新间隔与 kernel 能力共同决定结果。发布前应与 BF16 基线比较 loss curve、梯度异常、吞吐和 checkpoint 恢复，而不是只对一个矩阵报误差。


In [ ]:
def recipe_artifact151(recipe, metrics):
    # canonical JSON 的摘要把数值策略与验收结果绑定在一起。
    payload = json.dumps({"recipe": recipe, "metrics": metrics}, sort_keys=True, separators=(",", ":"))
    return {"payload": payload, "sha256": hashlib.sha256(payload.encode()).hexdigest()}

recipe151 = {"forward": "E4M3", "backward": "E5M2", "history": 16, "interval": 2, "margin": 1}
metrics151 = {"rel_rmse": rel_rmse151(ref151, out151), "saturation": diag_small151["saturation"]}
artifact151 = recipe_artifact151(recipe151, metrics151)
assert len(artifact151["sha256"]) == 64
assert json.loads(artifact151["payload"])["recipe"]["forward"] == "E4M3"
assert metrics151["rel_rmse"] < 0.12 and metrics151["saturation"] <= 0.01


## 面试总结

1. 先说 E4M3 精度更高、E5M2 范围更大，再说明具体边界取决于标准变体。
2. FP8 recipe 的核心状态包括 scale、amax history、margin、更新间隔和高精度主副本。
3. 数值验收同时看饱和、underflow、相对误差、训练曲线和恢复等价；性能必须在目标硬件真实 kernel 上测。
4. 教学量化器验证的是舍入与缩放逻辑，不生成可直接部署的 FP8 bitstream。

延伸阅读：[FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433)、[FP8-LM](https://arxiv.org/abs/2310.18313)、[NVIDIA Transformer Engine FP8 Primer](https://docs.nvidia.com/deeplearning/transformer-engine/user-guide/examples/fp8_primer.html)。
